# Installing Infiniband CNI in Kubernetes

This notebook shows the steps to install an Infiniband \
capable CNI in on the Heydar Cluster.

This guide is based on their and the documentation \
of the subprocess, which will be mentioned accordingly.

To be use this Notebook you need to have the jupyter 
bash kernel installed. Then you can run this:

```bash
python3 -m notebook --kernel bash --no-browser --port=8489 --ip=* --NotebookApp.password='sha256:e0976e673a12:3450891fc9a4a39745e034f37356eb7fdb4f689eaaf3ac9fb2e8320d3bca01f8'
```

First we check wether we are on the right host and \
if an Infiniband device is available.


In [9]:
hostname 

lspci | grep Infini

node19.lab.example
08:00.0 Infiniband controller: Mellanox Technologies MT27800 Family [ConnectX-5]


## Enabeling SR-IOV

To enable SR-IOV we can use the following command, \
as long as we have installed it before:

And then we take the deviceid of the Infiniband device \
and enable it with the following command:


In [1]:
if [ $? -ne 0 ]; then
    apt install -y mstflint

sudo mstconfig -d 0000:08:00.0 set SRIOV_EN=1 NUM_OF_VFS=8

After confirming the selection, \
the system needs to be rebooted.

```bash
reboot
```

### Testing

After the reboot the device should be able to \
create new VFs. We can test this with the following command:


In [10]:
echo -e "Before:\n"

echo -e " \nlspci"
lspci | grep Infini

echo "ip link"
ip link show | grep ib


echo -e "\n#################################### \n"
    sudo bash -c "echo 2 > /sys/class/net/ibs1/device/sriov_numvfs"

echo "After: "

echo -e " \nlspci"
lspci | grep Infini

echo -e "\nip link"
    ip link show | grep ib




Before:

 
lspci
08:00.0 Infiniband controller: Mellanox Technologies MT27800 Family [ConnectX-5]
ip link
8: ibs1: <BROADCAST,MULTICAST,UP,LOWER_UP> mtu 2044 qdisc mq state UP mode DEFAULT group default qlen 256
    link/infiniband 00:00:10:29:fe:80:00:00:00:00:00:00:04:09:73:ff:ff:d9:14:08 brd 00:ff:ff:ff:ff:12:40:1b:ff:ff:00:00:00:00:00:00:ff:ff:ff:ff
    altname ibp8s0

#################################### 

After: 
 
lspci
08:00.0 Infiniband controller: Mellanox Technologies MT27800 Family [ConnectX-5]
08:00.1 Infiniband controller: Mellanox Technologies MT27800 Family [ConnectX-5 Virtual Function]
08:00.2 Infiniband controller: Mellanox Technologies MT27800 Family [ConnectX-5 Virtual Function]

ip link
8: ibs1: <BROADCAST,MULTICAST,UP,LOWER_UP> mtu 2044 qdisc mq state UP mode DEFAULT group default qlen 256
    link/infiniband 00:00:10:29:fe:80:00:00:00:00:00:00:04:09:73:ff:ff:d9:14:08 brd 00:ff:ff:ff:ff:12:40:1b:ff:ff:00:00:00:00:00:00:ff:ff:ff:ff
    vf 0     link/infiniband 00:0

###  🎉 It works 🎉

---

### Cross cluster communication

Now we set up the Infiniband connectors on all of the nodes. \
And test the connection between the nodes.


In [44]:
# Pinging another node

sudo  ibping -c 5 -C     -P 1 -L 26

Pong from node18.lab.example.(none) (Lid 26): time 0.032 ms
Pong from node18.lab.example.(none) (Lid 26): time 0.027 ms
Pong from node18.lab.example.(none) (Lid 26): time 0.026 ms
Pong from node18.lab.example.(none) (Lid 26): time 0.027 ms
Pong from node18.lab.example.(none) (Lid 26): time 0.026 ms

--- node18.lab.example.(none) (Lid 26) ibping statistics ---
5 packets transmitted, 5 received, 0% packet loss, time 5000 ms
rtt min/avg/max = 0.026/0.027/0.032 ms


### Tagging nodes

Now that we know which nodes are Infiniband capable, \
we should tag them accordingly.So we know where to \
deploy the Infiniband CNI.

These are the nodes we know are Infiniband capable:

 - Heydar 20: port_lid:15
 - Heydar 19: port_lid:28
 - Heydar 18: port_lid:26
 - Heydar 17: port_lid:17
 - Heydar 13: port_lid:27


In [14]:
NODES=("heydar20" "heydar19" "heydar18" "heydar17" "heydar13")
postfix="proxy.example"

echo "
Setting the labels:
"

# setting the labels
for node in "${NODES[@]}"
do
  kubectl label nodes "${node}${postfix}" ib=true --overwrite
done

echo "
Checking the labels:
"

# checking the labels
kubectl get nodes --show-labels  | grep ib=true


Setting the labels:

node/node20.lab.example not labeled
node/node19.lab.example not labeled
node/node18.lab.example not labeled
node/node17.lab.example not labeled
node/node13.lab.example not labeled

Checking the labels:

node13.lab.example   Ready    <none>          276d   v1.27.4   beta.kubernetes.io/arch=amd64,beta.kubernetes.io/os=linux,ib=true,kubernetes.io/arch=amd64,kubernetes.io/hostname=node13.lab.example,kubernetes.io/os=linux
node17.lab.example   Ready    <none>          276d   v1.27.4   beta.kubernetes.io/arch=amd64,beta.kubernetes.io/os=linux,ib=true,kubernetes.io/arch=amd64,kubernetes.io/hostname=node17.lab.example,kubernetes.io/os=linux
node18.lab.example   Ready    <none>          276d   v1.27.4   beta.kubernetes.io/arch=amd64,beta.kubernetes.io/os=linux,ib=true,kubernetes.io/arch=amd64,kubernetes.io/hostname=node18.lab.example,kubernetes.io/os=linux
node19.lab.example   Ready    <none>          276d   v1.27.4   beta.kubernetes.io/arch=amd64,beta.kubernetes.io/os=lin

## Setting up the CNI

As the basis for this setup we are going to use \
the Multus CNI \footcite{multusK8snetworkplumbingwgMultuscni2024}, which is a meta CNI that allows us to \
use multiple CNIs at the same time. Creating more than \
one veth pair per pod, as discribed in the section \Nameref{Container networking}.

First of all, multus needs a default \ac{CNI} to be already installed. \
This we have done during the previous project, where we installed the \
Flannel CNI \footcite{joneckerthProjectPaperRepo} which we are going to use as the default CNI.

### Installing Multus   


In [8]:
# getting the file if not already there

if [ ! -f "multus-daemonset-thick.yml" ]; then
  wget https://raw.githubusercontent.com/k8snetworkplumbingwg/multus-cni/master/deployments/multus-daemonset-thick.yml
fi

kubectl apply -f multus-daemonset-thick.yml

kubectl get pods -n kube-system | grep multus


# Now lets start

customresourcedefinition.apiextensions.k8s.io/network-attachment-definitions.k8s.cni.cncf.io unchanged
clusterrole.rbac.authorization.k8s.io/multus unchanged
clusterrolebinding.rbac.authorization.k8s.io/multus unchanged
serviceaccount/multus unchanged
configmap/multus-daemon-config unchanged
daemonset.apps/kube-multus-ds unchanged
kube-multus-ds-2bdl9                                1/1     Running   0                 7m15s
kube-multus-ds-5x2dd                                1/1     Running   0                 7m15s
kube-multus-ds-5zb9m                                1/1     Running   0                 7m15s
kube-multus-ds-6ddv7                                1/1     Running   0                 7m15s
kube-multus-ds-6fp6b                                1/1     Running   0                 7m15s
kube-multus-ds-9z7bx                                1/1     Running   0                 7m15s
kube-multus-ds-bx5r6                                1/1     Running   0                 7m15s
kube-mult

### Getting the Infiniband CNI

With the nodes now able to speak to the infiniband devices, \
and the cluster to add more than once NIC to a pod, we can now \
we should now enable the containers to use the Infiniband devices.

This we can do by installing the ib-sriov-cni \footcite{K8snetworkplumbingwgIbsriovcniInfiniBand}:

```bash
cd 

if [ ! -d "ib-sriov-cni" ]; then
  git clone https://github.com/k8snetworkplumbingwg/ib-sriov-cni.git
fi

# now we also need to install golang if not already there
if [ ! -d "/usr/local/go" ]; then
  wget https://go.dev/dl/go1.22.2.linux-amd64.tar.gz
  sudo tar -C /usr/local -xzf go1.22.2.linux-amd64.tar.gz
  export PATH=$PATH:/usr/local/go/bin
  export GOPATH=$HOME/go
  export PATH=$PATH:$GOPATH/bin
fi

cd ib-sriov-cni
make 
```

In [20]:
  export PATH=$PATH:/usr/local/go/bin
  export GOPATH=$HOME/go
  export PATH=$PATH:$GOPATH/bin

## Adding the Infiniband CNI to the cluster

If the build was successful the CNI should be in the `./build` directory. \
From there we can copy the CNI to the cluster and put it in the `/opt/cni/bin` directory.

But as we dont want to do this manually, for every node, we can use a DaemonSet to do this for us.


In [25]:
# But first we need to build the image  
DOCKERFILE=Dockerfile TAG=node20.lab.example:31320/ib-sriov-cni make image 

Building Docker image...
## Build conatiner image
Emulate Docker CLI using podman. Create /etc/containers/nodocker to quiet msg.
[1/2] STEP 1/7: FROM golang:1.20-alpine AS builder
? Please select an image: 
  ▸ docker.io/library/golang:1.20-alpine
    node20.lab.example:31320/golang:1.20-alpine
ecorp.net:31320/golang:1.20-alpine
[2/2] STEP 1/6: FROM alpine:3.18.2
Error: error creating build container: ^C
make: *** [Makefile:129: image] Error 125


In [28]:
# Then we push the image to the registry
sudo docker push node20.lab.example:31320/ib-sriov-cni

Emulate Docker CLI using podman. Create /etc/containers/nodocker to quiet msg.
Getting image source signatures
Copying blob f0b83cc7dff9 [--------------------------------------] 0.0b / 0.0b
Copying blob f0b83cc7dff9 skipped: already exists  
Copying blob f0b83cc7dff9 skipped: already exists  
Copying blob 78a822fe2a2d [--------------------------------------] 0.0b / 0.0b
Copying blob f0b83cc7dff9 skipped: already exists  
Copying blob 78a822fe2a2d skipped: already exists  
Copying blob f0b83cc7dff9 skipped: already exists  
Copying blob 78a822fe2a2d skipped: already exists  
Copying blob f61b42c0ae5c [--------------------------------------] 0.0b / 0.0b
Copying blob f0b83cc7dff9 skipped: already exists  
Copying blob 78a822fe2a2d skipped: already exists  
Copying blob f61b42c0ae5c skipped: already exists  
Copying config 437a7ca1f0 [--------------------------------------] 0.0b / 1.7KiB
Copying config 437a7ca1f0 [--------------------------------------] 0.0b / 1.7KiB
Copying config 437a7ca

Now we edit the daemonset which will distribute the CNI to all nodes:
we simply swap the name of the container with the one we just built.

And add the nodeSelector to the DaemonSet, so it only gets deployed on 
the nodes we tagged with ib=true before.

```yaml
---
# Before applying this file, make sure to
# Remove all comments and empty lines
apiVersion: apps/v1
kind: DaemonSet
metadata:
  name: kube-ib-sriov-cni-ds-amd64
  namespace: kube-system
  labels:
    tier: node
    app: ib-sriov-cni
spec:
  selector:
    matchLabels:
      name: ib-sriov-cni
  template:
    metadata:
      labels:
        tier: node
        app: ib-sriov-cni
        name: ib-sriov-cni
    spec:
      hostNetwork: true
      nodeSelector:
        ib: "true"    # <-- Adding this line to select the ib nodes
      tolerations:
        - key: node-role.kubernetes.io/master
          operator: Exists
          effect: NoSchedule
      containers:
        - name: kube-ib-sriov-cni
        #   image: ghcr.io/k8snetworkplumbingwg/ib-sriov-cni       <-- Replacing this line
            image: node20.lab.example:31320/ib-sriov-cni  # <-- with this line to use our image
          imagePullPolicy: IfNotPresent
          securityContext:
            privileged: true
          resources:
            requests:
              cpu: "100m"
              memory: "50Mi"
            limits:
              cpu: "100m"
              memory: "50Mi"
          volumeMounts:
            - name: cnibin
              mountPath: /host/opt/cni/bin
      volumes:
        - name: cnibin
          hostPath:
            path: /opt/cni/bin
```

In [18]:
# Now we can deploy the DaemonSet

kubectl apply -f deployment/ib-sriov-cni-daemonset.yaml

# Now we can check if the pods are running on the right nodes
kubectl -n kube-system get pods -o=custom-columns=NAME:.metadata.name,NAMESPACE:.metadata.namespace,NODE:.spec.nodeName | grep kube-ib-sriov-cni 

daemonset.apps/kube-ib-sriov-cni-ds-amd64 unchanged
kube-ib-sriov-cni-ds-amd64-59qnq                    kube-system   node18.lab.example
kube-ib-sriov-cni-ds-amd64-82wx6                    kube-system   node13.lab.example
kube-ib-sriov-cni-ds-amd64-k8gs5                    kube-system   node20.lab.example
kube-ib-sriov-cni-ds-amd64-l86cd                    kube-system   node17.lab.example
kube-ib-sriov-cni-ds-amd64-wdzkb                    kube-system   node19.lab.example


### Configuration

Now we need to configure the CNI to use the Infiniband devices. \
The template provided by the ib-sriov-cni repository is a good starting point.


```yaml

apiVersion: "k8s.cni.cncf.io/v1"
kind: NetworkAttachmentDefinition
metadata:
  name: ib-sriov-network
  annotations:
    k8s.v1.cni.cncf.io/resourceName: mellanox.com/mlnx_sriov_rdma_ib
spec:
  config: '{
    "cniVersion": "0.3.1",
    "name": "InfinibandNetwork",
    "type": "ib-sriov",
    "deviceID": "0000:08:00.0",
    "link_state": "enable",
    "rdmaIsolation": true,
    "ibKubernetesEnabled": false,
    "ipam": {
      "type": "host-local",
      "subnet": "10.1.34.0/24"
    }
  }'

```

We also need to let the CNI know which devices are available. \
This we can do by creating a device plugin:

```yaml
apiVersion: v1
kind: ConfigMap
metadata:
  name: ib-sriov-cni-config
  namespace: kube-system
data:
  config.json: |
    {
        "resourceList": [{
                "resourcePrefix": "mellanox.com",
                "resourceName": "mlnx_sriov_rdma_ib",
                "selectors": {
                    "isRdma": true,
                    "vendors": ["15b3"],
                    "devices": ["1017", "1018"],
                    "drivers": ["mlx5_core"]
                }
            }
        ]
    }
```
#### Testing

Before rolling it out to the whole cluster, we should test the CNI on a single node. \
This we can do by creating a pod with the Infiniband CNI.

First we add the configs to the local node:



In [25]:
sudo bash -c "cat > /etc/cni/net.d/10-ib-sriov.conf <<EOF
{
    "cniVersion": "0.3.1",
    "name": "InfinibandNetwork",
    "type": "ib-sriov",
    "deviceID": "0000:08:00.0",
    "link_state": "enable",
    "rdmaIsolation": true,
    "ibKubernetesEnabled": false,
    "ipam": {
      "type": "host-local",
      "subnet": "10.1.34.0/24"
    } 
  }

EOF"

kubectl apply -f ib-sriov-cni-configmap.yaml

configmap/ib-sriov-cni-config created


Now we can create a pod with the Infiniband CNI:

```yaml
apiVersion: v1
kind: Pod
metadata:
  generateName: ib-test-
  annotations:
    k8s.v1.cni.cncf.io/networks: '[{"name": "InfinibandNetwork"}]'
spec:
  restartPolicy: OnFailure
  containers:
    - image: mellanox/rping-test
      imagePullPolicy: IfNotPresent
      name: my-test-ctr
      securityContext:
        capabilities:
          add: [ "IPC_LOCK" ]
      command:
        - sh
        - -c
        - |
          sleep 1000000
      resources:
        limits:
          mellanox.com/mlnx_sriov_rdma_ib: 1
        requests:
          mellanox.com/mlnx_sriov_rdma_ib: 1
```

And then run and check the pod:


In [28]:
kubectl apply -f tespod.yaml

pod/ib-test created


In [2]:
## Creating the certificates for the webhook

mkdir -p ./test/certs && cd ./test/certs

openssl genrsa -out ca.key 2048
openssl req -x509 -new -nodes -key ca.key -subj "/CN=webhook-ca" -days 10000 -out ca.crt

# Generate the webhook server key and certificate signing request (CSR)
openssl genrsa -out webhook.key 2048
openssl req -new -key webhook.key -subj "/CN=webhook-server.webhook-namespace.svc" -out webhook.csr

# Sign the CSR with the CA key to get the server certificate
openssl x509 -req -in webhook.csr -CA ca.crt -CAkey ca.key -CAcreateserial -out webhook.crt -days 10000
  




certificatesigningrequest.certificates.k8s.io/operator-webhook.sriov-network-operator.svc created
certificatesigningrequest.certificates.k8s.io/network-resources-injector.sriov-network-operator.svc created


In [9]:

kubectl certificate approve operator-webhook.sriov-network-operator.svc
kubectl certificate approve network-resources-injector.sriov-network-operator.svc


certificatesigningrequest.certificates.k8s.io/operator-webhook.sriov-network-operator.svc approved
certificatesigningrequest.certificates.k8s.io/network-resources-injector.sriov-network-operator.svc approved


In [10]:
# todo